In [ ]:
using NBInclude, LinearAlgebra, MAT
using Serialization

In [ ]:
@nbinclude("LRP.ipynb") # a solution in LRP form in V(4,4,4|48) found by Dumas, Pernet and Sedoglavic

In [ ]:
# ============================================================
# Known tangent directions: de Groote + layer scaling
# Rows of G_known use the same variable order as columns of tangent_matrix:
#     [vec(U_1); vec(V_1); vec(W_1); ...; vec(U_L); vec(V_L); vec(W_L)].
# ============================================================

function build_tangent_matrix(sol; dtype::Type = Float64)
    @assert !isempty(sol) "sol must be nonempty"

    L = length(sol)
    U1, V1, W1 = sol[1]

    # This de Groote implementation assumes the present case:
    # U_k, V_k, W_k are all m x m matrices.
    m = size(U1, 1)
    @assert size(U1) == (m, m)
    @assert size(V1) == (m, m)
    @assert size(W1) == (m, m)

    nvars = L * 3 * m * m
    cols = Vector{Vector{dtype}}()

    function var_index(k::Int, kind::Symbol, i::Int, j::Int)
        base = (k - 1) * 3 * m * m
        offset = if kind === :u
            0
        elseif kind === :v
            m * m
        elseif kind === :w
            2 * m * m
        else
            error("unknown variable kind: $kind")
        end
        return base + offset + i + (j - 1) * m
    end

    function add_matrix_delta!(col, k::Int, kind::Symbol, Δ)
        for j in axes(Δ, 2), i in axes(Δ, 1)
            val = Δ[i, j]
            if val != 0
                col[var_index(k, kind, i, j)] += convert(dtype, val)
            end
        end
        return nothing
    end

    Tm = eltype(U1)
    Es = Matrix{Tm}[]
    for a in 1:m, b in 1:m
        E = zeros(Tm, m, m)
        E[a, b] = one(Tm)
        push!(Es, E)
    end

    Z = zeros(Tm, m, m)

    # de Groote A:
    #     δU = A U,  δV = 0,    δW = -W A
    for E in Es
        col = zeros(dtype, nvars)
        for k in 1:L
            U, V, W = sol[k]
            @assert size(U) == (m, m)
            @assert size(V) == (m, m)
            @assert size(W) == (m, m)

            add_matrix_delta!(col, k, :u, E * U)
            add_matrix_delta!(col, k, :v, Z)
            add_matrix_delta!(col, k, :w, -W * E)
        end
        push!(cols, col)
    end

    # de Groote B:
    #     δU = -U B, δV = B V, δW = 0
    for E in Es
        col = zeros(dtype, nvars)
        for k in 1:L
            U, V, W = sol[k]

            add_matrix_delta!(col, k, :u, -U * E)
            add_matrix_delta!(col, k, :v, E * V)
            add_matrix_delta!(col, k, :w, Z)
        end
        push!(cols, col)
    end

    # de Groote C:
    #     δU = 0,    δV = -V C, δW = C W
    for E in Es
        col = zeros(dtype, nvars)
        for k in 1:L
            U, V, W = sol[k]

            add_matrix_delta!(col, k, :u, Z)
            add_matrix_delta!(col, k, :v, -V * E)
            add_matrix_delta!(col, k, :w, E * W)
        end
        push!(cols, col)
    end

    # Layer scaling, for each layer k:
    #     a_k: δU = U, δV = 0, δW = -W
    #     b_k: δU = 0, δV = V, δW = -W
    for k in 1:L
        U, V, W = sol[k]

        col_a = zeros(dtype, nvars)
        add_matrix_delta!(col_a, k, :u, U)
        add_matrix_delta!(col_a, k, :w, -W)
        push!(cols, col_a)

        col_b = zeros(dtype, nvars)
        add_matrix_delta!(col_b, k, :v, V)
        add_matrix_delta!(col_b, k, :w, -W)
        push!(cols, col_b)
    end

    return hcat(cols...)
end

In [ ]:
Ts = build_tangent_matrix(LRP) #Here, Ts is the tagent basis matrix T(s), where s is the solution stored in LRP.ipynb

In [ ]:
println("size(Ts) = ", size(Ts))

In [ ]:
rank_known = rank(Ts; atol = 1e-9, rtol = 1e-9)

In [ ]:
println("rank(Ts) = ", rank_known)

In [ ]:
# check if exists
if !(@isdefined Ts)
    error("Ts does not exist, run ：Ts = build_tangent_matrix(L)")

end

In [ ]:
# -------------------------
# save as  .jls data
# -------------------------

serialize("Ts.jls", Ts)

In [ ]:
# -------------------------
# 2. optianal: save as .jl data
# -------------------------

function write_one_matrix_as_jl(
    filename::AbstractString,
    name::AbstractString,
    A::AbstractMatrix;
    keep_negative_zero::Bool = false,
)
    @assert occursin(r"^[A-Za-z_][A-Za-z_0-9]*$", name) "变量名 $name 不是合法 Julia 变量名"

    T = eltype(A)
    m, n = size(A)

    open(filename, "w") do io
        println(io, "# Generated Julia matrix file")
        println(io, "# Load by: include(\"$filename\")")
        println(io)
        println(io, "$name = zeros($T, $m, $n)")

        for j in 1:n
            for i in 1:m
                a = A[i, j]

                # 默认不保存 0.0 和 -0.0，因为数值上它们都是 0
                # 若需要严格保留 -0.0，可设置 keep_negative_zero = true
                if !iszero(a) || (keep_negative_zero && a isa AbstractFloat && signbit(a))
                    println(io, "$name[$i, $j] = $(repr(a))")
                end
            end
        end
    end

    return filename
end

In [ ]:
write_one_matrix_as_jl("Ts.jl", "Ts", Ts)

In [ ]:
# -------------------------
# Optianal: save as matlab data
# -------------------------

matwrite("Ts.mat", Dict("Ts" => Ts))

In [ ]:
# optional: in the following, check J(s)T(s)=0

In [ ]:
# transform s into vector form
L0 = hcat(([LRP[i][1] LRP[i][2] LRP[i][3]] for i in 1:48)...)
L1 = reshape(L0, 2304, 1)
L=Float64.(L1)

In [ ]:
# generate J(s)
function Jacobrent(x, m::Integer, n::Integer, p::Integer, r::Integer)
    A = m * n
    B = n * p
    C = p * m
    s = A + B + C

    X = reshape(x, s, r)

    J = Matrix{eltype(x)}(undef, A * B * C, r * s)

    IA = Matrix{eltype(x)}(I, A, A)
    IB = Matrix{eltype(x)}(I, B, B)
    IC = Matrix{eltype(x)}(I, C, C)

    for t in 1:r
        a = reshape(X[1:A, t], A, 1)
        b = reshape(X[A+1:A+B, t], B, 1)
        c = reshape(X[A+B+1:A+B+C, t], C, 1)

        block = hcat(
            kron(kron(IA, b), c),
            kron(kron(a, IB), c),
            kron(kron(a, b), IC)
        )

        cols = ((t - 1) * s + 1):(t * s)
        J[:, cols] = block
    end

    return J
end

In [ ]:
Js=Jacobrent(L,4,4,4,48)  # the jacobian J(s)

In [ ]:
norm(Js*Ts)